In [6]:
!pip -q install langchain-huggingface langchain sentence_transformers PyPDF2 pycryptodome tiktoken faiss-gpu

In [94]:
import os
import re
from PyPDF2 import PdfReader
from langchain.text_splitter import CharacterTextSplitter
from langchain import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

from vertexai.generative_models import GenerationConfig, GenerativeModel, Content, GenerativeModel, Part  
from google.cloud import aiplatform  
from IPython.display import Markdown


In [8]:
pdf_reader = PdfReader('data/naf23.pdf')
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

In [9]:
def process_text(text, chunk_size=1000, chunk_overlap=100):
    text_splitter = CharacterTextSplitter(
        separator="\n",
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len
    )
    chunks = text_splitter.split_text(text)
    embedding_model = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
    knowledgeBase = FAISS.from_texts(chunks, embedding_model)
    return knowledgeBase

knowledgeBase = process_text(text)

In [119]:
def search(query):
    return knowledgeBase.similarity_search(query, k=1)[0].page_content

In [120]:
search('total revenue')

'Total Revenue 979,651           -                        979,651           \nNet Assets Released from Restrictions 975,954           (975,954)          -                        \nTotal Support and Revenue 3,546,026       638,761           4,184,787       \nExpenses\nProgram services\nResearch 1,574,354       -                        1,574,354       \nEducation and service 1,146,980       -                        1,146,980       \nDrug Development Collaborative 1,188,427       -                        1,188,427       \nTotal Program Services 3,909,761       -                        3,909,761       \nSupporting services\nManagement and general 538,079           -                        538,079           \nFundraising 548,618           -                        548,618           \nTotal Supporting Services 1,086,697       -                        1,086,697       \nTotal Expenses 4,996,458       -                        4,996,458'

In [13]:
model = GenerativeModel("gemini-1.5-pro-001")

In [14]:
chat = model.start_chat()
response = chat.send_message(
    """how many superbowls has Rams won
"""
)
print(response.text)

The Rams have won **two** Super Bowls since the franchise began:

* **Super Bowl XXXIV (2000):** Defeated the Tennessee Titans 23-16
* **Super Bowl LVI (2022):** Defeated the Cincinnati Bengals 23-20 

While the franchise has a longer history with a previous stint in St. Louis, those two championships are the only Super Bowl victories for the Rams. 



In [95]:
class Agent:
    def __init__(self, client, system_prompt):
        self.client=client # the gemini model
        self.system_prompt = system_prompt
        self.history = []
            
        if self.system_prompt:
            self.history.append(self.create_Content(role='user', text=self.system_prompt))

    def __call__(self, message):        
        result = self.execute(message)
        self.history.append(self.create_Content(role='user', text=message))
        self.history.append(self.create_Content(role='user', text=result))
        return result
    
    def create_Content(self, role, text):
        return Content(role=role, parts=[Part.from_text(text)])
    
    def execute(self, message):
        chat = model.start_chat(history=self.history)
        response = chat.send_message(content=message, generation_config = GenerationConfig(temperature=1.0))
        return response.text


In [137]:
system_prompt = """
you are a financial expert and an underwriter working for commerical banking of a bank.
you have a knowledge base, for any specific information you do not know, you can search the knowledge base, the function is defined later.

You run in a loop of Thought, Action, PAUSE, Observation.
At the end of the loop you output an Answer
Use Thought to describe your thoughts about the question you have been asked.
Use Action to run one of the actions available to you - then return PAUSE.
Observation will be the result of running those actions.

Your available actions are:

search:
e.g. search: total income
Runs a serach function in the knowledge base and return relevant information as chunks of text.


Example session (This is only an example):

Question: analyze financial performance of Apple Inc and potential drivers?
Thought: to analyse finanial performance, I need to know Apple's total revenue, so I need to search its total revenue
Action: search: Apple Inc total revenue
PAUSE

You will be called again with this:

Observation: 'Total Revenue 123,456...'

Thought: to know the potential drivers, I need to know the buisness composition of Apple, so I need to search its buisness composition
Action: search: Apple Inc buisness composition
PAUSE

You will be called again with this:

Observation: 'Apple's business composition includes iPhone: Major revenue driver. Mac: Laptops and desktops...'

If you have the answer, output it as the Answer.

Answer: Apple's financial performance is strong with a total revenue of 123,456, which is drived by iPhone sales.

Now it's your turn:
"""

In [138]:
class ReactExcecutor:
    def __init__(self, client, tools, system_prompt, max_iterations=10):

        self.client = client
        self.tools = tools
        self.system_prompt = system_prompt
        self.max_iterations = max_iterations

    def run(self, query):
        agent = Agent(client = self.client, system_prompt=self.system_prompt)
        next_prompt = query
        i = 0
        while i < self.max_iterations:
            i += 1
            result = agent(next_prompt)
            print("-"*80)
            print(result)
            print()

            if "PAUSE" in result and "Action" in result:
                action = re.findall(r"Action: ([a-z_]+): (.+)", result, re.IGNORECASE)
                chosen_tool = action[0][0]
                arg = action[0][1]

                if chosen_tool in self.tools:
                    result_tool = eval(f"{chosen_tool}('{arg}')")                
                    next_prompt = f"Observation: {result_tool}"                    
                else:
                    next_prompt = "Observation: Tool not found"
                print("-"*80)
                print(next_prompt)
                print()
                continue

            if "Answer" in result:
                break

In [139]:
client = GenerativeModel("gemini-1.5-pro-001")
reactExcecutor = ReactExcecutor(client=client, tools=["search"], system_prompt=system_prompt)

In [143]:
reactExcecutor.run(query="total revenue in 2023")

--------------------------------------------------------------------------------
Thought: I need to find the total revenue of what in 2023? I need more information. 
Action: None
PAUSE 




IndexError: list index out of range

In [141]:
reactExcecutor.run(query="what is the total revenue")

--------------------------------------------------------------------------------
Thought: I need to search the knowledge base to find the total revenue. 
Action: search: total revenue
PAUSE 


--------------------------------------------------------------------------------
Observation: Total Revenue 979,651           -                        979,651           
Net Assets Released from Restrictions 975,954           (975,954)          -                        
Total Support and Revenue 3,546,026       638,761           4,184,787       
Expenses
Program services
Research 1,574,354       -                        1,574,354       
Education and service 1,146,980       -                        1,146,980       
Drug Development Collaborative 1,188,427       -                        1,188,427       
Total Program Services 3,909,761       -                        3,909,761       
Supporting services
Management and general 538,079           -                        538,079           
Fundraising

In [140]:
reactExcecutor.run(query="what is the total revenue of National Ataxia Foundation in 2023")

--------------------------------------------------------------------------------
Thought: To answer the question, I need to find the total revenue of National Ataxia Foundation in 2023. I will search the knowledge base for this information. 
Action: search: National Ataxia Foundation total revenue 2023
PAUSE 


--------------------------------------------------------------------------------
Observation: Statements of Cash Flows
For the Years Ended December 31, 2023 and 2022
See Independent Auditor's Report and Notes to the Financial Statements.
11 
 National Ataxia Foundation  
Notes to the Financial Statements  
December 31, 2023  and 2022  
 
Note 1:  Summary of Significant Accounting Policies  
 
A. Nature of Activities  
 
National Ataxia Foundation  (the Foundation) was incorporated as a Minnesota nonprofit corporation in 1957. The 
Foundation’s mission is to accelerate the development of treatments and a cure while working to improve the lives of 
those living with Ataxia.  
 
To

In [142]:
query="What caused the significant change in net assets with donor restrictions between 2022 and 2023?"
reactExcecutor.run(query=query)

--------------------------------------------------------------------------------
Thought: I need to understand how net assets with donor restrictions changed, so I need to see the values for both years. I will search for those values. 
Action: search: net assets with donor restrictions 2022 and 2023
PAUSE 


--------------------------------------------------------------------------------
Observation: December 31, 2023  and 2022  
 
Note 10: Liquidity and Availability of Financial Assets  
 
Financial assets available for general expenditure, that is, without donor or other restrictions limiting their use, within o ne 
year of the statement of financial position dates, comprise the following:  
 
2023 2022
Cash and cash equivalents 1,008,716$     1,969,164$     
Investments 2,788,623       2,707,996       
Accounts receivable 6,438               55,331             
Total Financial Assets Available Within One Year 3,803,777       4,732,491       
Less: Amounts unavailable for general exp

In [144]:
query="What was the operating lease liability at the end of 2023, and how does the lease structure affect NAF’s financial obligations?"
reactExcecutor.run(query=query)

--------------------------------------------------------------------------------
Thought: To answer this question, I need to find the operating lease liability specifically at the end of 2023. I also need to understand how NAF structures its leases and how that impacts their overall financial obligations.

Action: search: NAF operating lease liability 2023 
PAUSE 


--------------------------------------------------------------------------------
Observation: for fixed monthly base payments starting at $2,563 per month and incrementally increasing to $2,864 per month until 
expiration in September  2025. The agreement also provides that the Foundation  is responsible for a proportional share of 
the property's incurred ownership taxes and operating expenses. The Foundation  has determined that this lease is an 
operating lease.  
 
The Foundation  adopted FASB ASC 842, effective January 1, 2022, using a modified retrospective approach. As a result, 
the Foundation  was required to recog

IndexError: list index out of range

In [147]:
query ="How did National Ataxia Foundation's deferred revenue change between 2022 and 2023, and what drove this change?"
reactExcecutor.run(query=query)

--------------------------------------------------------------------------------
Thought: To answer this question, I need to find the deferred revenue for National Ataxia Foundation in 2022 and 2023. 
Action: search: National Ataxia Foundation deferred revenue 2022 and 2023
PAUSE 


--------------------------------------------------------------------------------
Observation: the years ended December 31, 2023  and 2022 , respectively. Deferred Drug D evelopment Collaborative revenue 
was $100,000  and $0 for the years ended December 31, 2023  and 2022 , respectively.  
 
The Foundation ’s other revenues are explicitly excluded from the scope of ASC Topic 606 and are not recorded in 
accordance with that standard.  All contributions are considered to be available for unrestricted use unless specifically 
restricted by the donor. The Board’s general policy is to allocate up to thirty percent of all restricted contributions for 
unrestricted use unless the contribution was solicited by the